In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Класс наивного байесовского классификатора
class NaiveBayesClassifier:
    def __init__(self, alpha=1):
        """
        alpha - параметр сглаживания Лапласа
        """
        self.alpha = alpha
        self.class_probs = {}
        self.feature_probs = defaultdict(dict)
        self.classes = None
        self.feature_values = {}
    
    def fit(self, X, y):
        """
        Обучение модели
        X - DataFrame с признаками
        y - Series с целевой переменной
        """
        self.classes = y.unique()
        n_samples = len(y)
        
        # 1. Рассчёт априорных вероятностей классов P(y)
        for c in self.classes:
            self.class_probs[c] = (y == c).sum() / n_samples
        
        # 2. Рассчёт условных вероятностей признаков P(x|y) с сглаживанием Лапласа
        for feature in X.columns:
            # Сохраняем уникальные значения признака
            self.feature_values[feature] = X[feature].unique()
            
            for c in self.classes:
                # Подмножество объектов класса c
                X_c = X[y == c]
                n_c = len(X_c)
                
                # Подсчёт частот значений признака в классе c
                value_counts = X_c[feature].value_counts()
                
                # Рассчёт вероятностей с сглаживанием Лапласа
                probs = {}
                for value in self.feature_values[feature]:
                    count = value_counts.get(value, 0)
                    # Формула сглаживания Лапласа: (count + alpha) / (n_c + alpha * n_values)
                    probs[value] = (count + self.alpha) / (n_c + self.alpha * len(self.feature_values[feature]))
                
                self.feature_probs[feature][c] = probs
    
    def predict_proba(self, X):
        """
        Предсказание вероятностей классов
        Возвращает массив вероятностей для каждого класса
        """
        n_samples = len(X)
        n_classes = len(self.classes)
        probas = np.zeros((n_samples, n_classes))
        
        for i in range(n_samples):
            for j, c in enumerate(self.classes):
                # Начинаем с априорной вероятности класса (в логарифмической шкале)
                log_prob = np.log(self.class_probs[c])
                
                # Добавляем логарифмы условных вероятностей признаков
                for feature in X.columns:
                    value = X.iloc[i][feature]
                    if value in self.feature_probs[feature][c]:
                        log_prob += np.log(self.feature_probs[feature][c][value])
                    else:
                        # Если значение не встречалось при обучении, используем сглаживание
                        log_prob += np.log(self.alpha / (len(self.feature_values[feature]) * self.alpha))
                
                probas[i, j] = np.exp(log_prob)
            
            # Нормализация вероятностей
            if probas[i].sum() > 0:
                probas[i] /= probas[i].sum()
        
        return probas
    
    def predict(self, X):
        """
        Предсказание классов
        """
        probas = self.predict_proba(X)
        predictions = []
        
        for i in range(len(X)):
            # Выбираем класс с максимальной вероятностью
            class_idx = np.argmax(probas[i])
            predictions.append(self.classes[class_idx])
        
        return predictions

# Функция для оценки качества модели
def evaluate_model(y_true, y_pred, y_pred_proba=None):
    """
    Вычисление метрик качества классификации
    """
    metrics = {}
    
    # 1. Accuracy (точность)
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    
    # 2. Precision, Recall, F1 для класса 'Infected'
    metrics['precision_infected'] = precision_score(y_true, y_pred, pos_label='Infected', zero_division=0)
    metrics['recall_infected'] = recall_score(y_true, y_pred, pos_label='Infected', zero_division=0)
    metrics['f1_infected'] = f1_score(y_true, y_pred, pos_label='Infected', zero_division=0)
    
    # 3. Precision, Recall, F1 для класса 'Not_infected'
    metrics['precision_not_infected'] = precision_score(y_true, y_pred, pos_label='Not_infected', zero_division=0)
    metrics['recall_not_infected'] = recall_score(y_true, y_pred, pos_label='Not_infected', zero_division=0)
    metrics['f1_not_infected'] = f1_score(y_true, y_pred, pos_label='Not_infected', zero_division=0)
    
    # 4. Confusion matrix
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred, labels=['Infected', 'Not_infected'])
    
    return metrics

def print_metrics(metrics):
    """
    Красивый вывод метрик
    """
    print("\n" + "="*60)
    print("МЕТРИКИ КАЧЕСТВА МОДЕЛИ")
    print("="*60)
    
    print(f"\nОбщая точность (Accuracy): {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    
    print("\n--- Для класса 'Infected' ---")
    print(f"Precision: {metrics['precision_infected']:.4f}")
    print(f"Recall:    {metrics['recall_infected']:.4f}")
    print(f"F1-score:  {metrics['f1_infected']:.4f}")
    
    print("\n--- Для класса 'Not_infected' ---")
    print(f"Precision: {metrics['precision_not_infected']:.4f}")
    print(f"Recall:    {metrics['recall_not_infected']:.4f}")
    print(f"F1-score:  {metrics['f1_not_infected']:.4f}")
    
    print("\nМатрица ошибок (Confusion Matrix):")
    print("                 Предсказано")
    print("                 Infected  Not_infected")
    cm = metrics['confusion_matrix']
    print(f"Истинно Infected     {cm[0,0]}         {cm[0,1]}")
    print(f"Истинно Not_infected {cm[1,0]}         {cm[1,1]}")

# Основной блок выполнения

# Чтение данных из CSV файла
# Предполагаем, что данные находятся в файле 'medical_data.csv'
# Создадим DataFrame из предоставленных данных

# Чтение данных из CSV файла
try:
    df = pd.read_csv('/Users/phil/GitHub/masters_degree_Roshchin_M25-555/dataset_diseases.csv')
    print("Данные успешно загружены из CSV файла")
    print(f"Размер данных: {df.shape}")
    print("\nПервые 5 строк данных:")
    print(df.head())
except FileNotFoundError:
    print("Файл не найден, используем встроенные данные")

# Разделение на признаки и целевую переменную
X = df[['Test', 'Age_Group']]
y = df['Status']

# Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

# Создание и обучение модели
print("\nОбучение модели Наивного Байеса...")
model = NaiveBayesClassifier(alpha=1)  # alpha=1 для сглаживания Лапласа
model.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Оценка качества
metrics = evaluate_model(y_test, y_pred, y_pred_proba)

# Вывод метрик
print_metrics(metrics)

# Вывод дополнительной информации о модели
print("\n" + "="*60)
print("ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ О МОДЕЛИ")
print("="*60)

print(f"\nАприорные вероятности классов:")
for cls, prob in model.class_probs.items():
    print(f"  P({cls}) = {prob:.4f}")

print(f"\nУсловные вероятности признаков:")
for feature in ['Test', 'Age_Group']:
    print(f"\nПризнак: {feature}")
    for cls in model.classes:
        print(f"  Для класса '{cls}':")
        for value, prob in model.feature_probs[feature][cls].items():
            print(f"    P({feature}={value}|{cls}) = {prob:.4f}")

# Пример предсказания для новых данных
print("\n" + "="*60)
print("ПРИМЕРЫ ПРЕДСКАЗАНИЙ ДЛЯ НОВЫХ ДАННЫХ")
print("="*60)

new_data = pd.DataFrame({
    'Test': ['Positive', 'Negative', 'Positive', 'Negative'],
    'Age_Group': ['Young', 'Old', 'Old', 'Young']
})

new_predictions = model.predict(new_data)
new_probas = model.predict_proba(new_data)

for i in range(len(new_data)):
    print(f"\nОбразец {i+1}:")
    print(f"  Признаки: Test={new_data.iloc[i]['Test']}, Age_Group={new_data.iloc[i]['Age_Group']}")
    print(f"  Предсказанный класс: {new_predictions[i]}")
    print(f"  Вероятности: Infected={new_probas[i, 0]:.4f}, Not_infected={new_probas[i, 1]:.4f}")

# Вывод статистики по данным
print("\n" + "="*60)
print("СТАТИСТИКА ПО ДАННЫМ")
print("="*60)

print(f"\nРаспределение классов в исходных данных:")
class_dist = y.value_counts()
for cls, count in class_dist.items():
    print(f"  {cls}: {count} ({count/len(y)*100:.1f}%)")

print(f"\nРаспределение признака 'Test':")
test_dist = X['Test'].value_counts()
for val, count in test_dist.items():
    print(f"  {val}: {count} ({count/len(X)*100:.1f}%)")

print(f"\nРаспределение признака 'Age_Group':")
age_dist = X['Age_Group'].value_counts()
for val, count in age_dist.items():
    print(f"  {val}: {count} ({count/len(X)*100:.1f}%)")

# Анализ важности признаков (на основе разности вероятностей)
print("\n" + "="*60)
print("АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ")
print("="*60)

for feature in ['Test', 'Age_Group']:
    print(f"\nПризнак: {feature}")
    # Рассчитаем, насколько различаются распределения для разных классов
    for value in model.feature_values[feature]:
        prob_infected = model.feature_probs[feature]['Infected'][value]
        prob_not_infected = model.feature_probs[feature]['Not_infected'][value]
        diff = abs(prob_infected - prob_not_infected)
        print(f"  {value}: P(Infected)={prob_infected:.4f}, P(Not_infected)={prob_not_infected:.4f}, разница={diff:.4f}")

Данные успешно загружены из CSV файла
Размер данных: (278, 3)

Первые 5 строк данных:
       Test Age_Group    Status
0  Positive     Young  Infected
1  Positive     Young  Infected
2  Positive     Young  Infected
3  Positive       Old  Infected
4  Positive       Old  Infected

Размер обучающей выборки: (222, 2)
Размер тестовой выборки: (56, 2)

Обучение модели Наивного Байеса...

МЕТРИКИ КАЧЕСТВА МОДЕЛИ

Общая точность (Accuracy): 0.7857 (78.57%)

--- Для класса 'Infected' ---
Precision: 0.8400
Recall:    0.7241
F1-score:  0.7778

--- Для класса 'Not_infected' ---
Precision: 0.7419
Recall:    0.8519
F1-score:  0.7931

Матрица ошибок (Confusion Matrix):
                 Предсказано
                 Infected  Not_infected
Истинно Infected     21         8
Истинно Not_infected 4         23

ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ О МОДЕЛИ

Априорные вероятности классов:
  P(Infected) = 0.5090
  P(Not_infected) = 0.4910

Условные вероятности признаков:

Признак: Test
  Для класса 'Infected':
    P(Test